In [1]:
import os
# GPU selection: set via env var GPU_INDEX or directly here (default '0')
# Example: export GPU_INDEX=1  (before starting the notebook)
gpu_index = os.getenv("GPU_INDEX", "0")
os.environ["CUDA_VISIBLE_DEVICES"] = gpu_index
print(f"Using GPU index: {gpu_index}")

Using GPU index: 0


In [2]:
import functools
import getpass
import json
from typing import List, Optional
import pandas as pd


# Settings
os.environ["QUERY_HUGGINGFACE"] = "True"  # if set to True, this runs the model locally
os.environ["HUGGINGFACE_MODEL"] = "Qwen/Qwen3-1.7B"  # the model to use
os.environ["HUGGINGFACE_N"] = "1"  # the number of answers to generate per prompt
os.environ["HUGGINGFACE_TEMPERATURE"] = "1"  # the model temperature to use
os.environ["TEMPLATE_DIR"] = "templates/"
os.environ["SQLITE_PATH"] = "huggingface_results_1-N.sqlite3"  # the path to the SQLite database file.

# GPU Optimization: Concurrent batch processing (improves utilization from ~40-60% to 80-95%)
os.environ["HUGGINGFACE_MAX_CONCURRENT"] = "4"  # max concurrent batch operations (tune based on GPU VRAM)
os.environ["HUGGINGFACE_BATCH_SIZE"] = "64"  # chunk size for batch inference (8-32 recommended)
# Tuning guide: 24GB+ VRAM: max_concurrent=8, batch_size=32 | 12-16GB: max_concurrent=4, batch_size=16 | 6-8GB: max_concurrent=2, batch_size=8

In [3]:
# IMPORTANT: Make sure you've run Cell 0 above first!
from fm_matcher.utils_huggingface import storage
from fm_matcher.utils_huggingface.models import Attribute, Parameters, Relation, Result, Side
from fm_matcher.utils_huggingface.prompt_building import PromptDesign, build_prompts
from fm_matcher.utils_huggingface.prompt_sending import process_prompt_list
from fm_matcher.utils_huggingface.prompt_postprocessing import postprocess_answers

# Only use 1-to-N mode
all_templates = ["oneToN"]
all_modes = [PromptDesign.oneToN]

/home/yuej18/miniconda3/envs/schema_eval/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
benchmark = pd.read_csv("benchmark/CMS-OMOP-ground_truth.csv") #replace
for side in ("source", "target"):
    benchmark[[f"{side}_schema", f"{side}_relation", f"{side}_attribute"]] = benchmark[side].str.split(".", expand=True)
    benchmark[side] = benchmark[side].str.lower()
benchmark["benchmark"] = True

relation_combinations = benchmark[["source_relation", "target_relation"]].drop_duplicates().values
print(relation_combinations)

[['beneficiarysummary' 'person']
 ['inpatientclaims' 'visit_occurrence']
 ['outpatientclaims' 'visit_occurrence']
 ['carrierclaims' 'visit_occurrence']
 ['inpatientclaims' 'measurement']
 ['outpatientclaims' 'measurement']
 ['carrierclaims' 'measurement']
 ['inpatientclaims' 'procedure_occurrence']
 ['outpatientclaims' 'procedure_occurrence']
 ['carrierclaims' 'procedure_occurrence']
 ['inpatientclaims' 'condition_occurrence']
 ['outpatientclaims' 'condition_occurrence']
 ['carrierclaims' 'condition_occurrence']
 ['beneficiarysummary' 'death']
 ['inpatientclaims' 'death']
 ['outpatientclaims' 'death']
 ['carrierclaims' 'death']
 ['beneficiarysummary' 'location']
 ['inpatientclaims' 'observation']
 ['outpatientclaims' 'observation']
 ['carrierclaims' 'observation']
 ['inpatientclaims' 'drug_exposure']
 ['outpatientclaims' 'drug_exposure']
 ['carrierclaims' 'drug_exposure']
 ['prescriptiondrugevents' 'drug_exposure']
 ['inpatientclaims' 'device_exposure']
 ['outpatientclaims' 'device_exp

In [5]:
@functools.cache
def get_description(schema: str, table: str, attribute: Optional[str] = None) -> str:
    if attribute:
        filename = f"{schema}-{table}-{attribute}.txt"
        print(filename)
    else:
        filename = f"{schema}-table-{table}.txt"
        print(filename)
    filename_filter = lambda f: f.lower() == filename.lower()
    try:
        filename = next(filter(filename_filter, os.listdir("schema_documentations")))
        with open(os.path.join("schema_documentations", filename), "r") as desc_file:
            description = desc_file.read()
        return description
    except StopIteration:
        return "No description available."


@functools.cache
def get_attributes(schema: str, table: str) -> List[str]:
    table_filter = lambda f: f.lower().startswith(f"{schema}-{table}-") and f.endswith(".txt")
    extract_attr_name = lambda f: f[len(f"{schema}-{table}-"):-len(".txt")]
    return [
        extract_attr_name(attr_file)
        for attr_file in filter(table_filter, os.listdir("schema_documentations/"))
    ]


@functools.cache
def get_relation(schema: str, table: str) -> Relation:
    schema, table = schema.lower(), table.lower()
    description = get_description(schema, table)
    attributes = [
        Attribute(
            name=attr_name.capitalize(),
            description=get_description(schema, table, attr_name),
        ) for attr_name in get_attributes(schema, table)
    ]

    side = Side.SOURCE if schema == "cms" else Side.TARGET #replace
    
    return Relation(
        name=table.capitalize(),
        side=side,
        description=description,
        attributes=attributes,
    )


parameters = [
    Parameters(
        source_relation=get_relation("cms", source), #replace
        target_relation=get_relation("omop", target),
    ) for source, target in relation_combinations
]
parameters = [storage.store_parameters(p) for p in parameters]

cms-table-beneficiarysummary.txt
cms-beneficiarysummary-bene_birth_dt.txt
cms-beneficiarysummary-bene_county_cd.txt
cms-beneficiarysummary-bene_death_dt.txt
cms-beneficiarysummary-bene_esrd_ind.txt
cms-beneficiarysummary-bene_hi_cvrage_tot_mons.txt
cms-beneficiarysummary-bene_hmo_cvrage_tot_mons.txt
cms-beneficiarysummary-bene_race_cd.txt
cms-beneficiarysummary-bene_sex_ident_cd.txt
cms-beneficiarysummary-bene_smi_cvrage_tot_mons.txt
cms-beneficiarysummary-benres_car.txt
cms-beneficiarysummary-benres_ip.txt
cms-beneficiarysummary-benres_op.txt
cms-beneficiarysummary-desynpuf_id.txt
cms-beneficiarysummary-medreimb_car.txt
cms-beneficiarysummary-medreimb_ip.txt
cms-beneficiarysummary-medreimb_op.txt
cms-beneficiarysummary-plan_cvrg_mos_num.txt
cms-beneficiarysummary-pppymt_car.txt
cms-beneficiarysummary-pppymt_ip.txt
cms-beneficiarysummary-pppymt_op.txt
cms-beneficiarysummary-sp_alzhdmta.txt
cms-beneficiarysummary-sp_chf.txt
cms-beneficiarysummary-sp_chrnkidn.txt
cms-beneficiarysummary-s

In [6]:
# Only define the oneToN template
templates = {
    "oneToN": [
        {
            "role": "user",
            "content": (
                "Act as a schema matcher for relational schemas. Your task is to create semantic matches"
                " that specify how the elements of the source schema and the target schema semantically"
                " correspond to one another. Two attributes semantically match if and only if there exists"
                " an invertible function that maps all values of one attribute to the other. First, I will"
                " input the name of an attribute from the source schema, a description of the attribute,"
                " the name of the relation in belongs to and a description of this relation. After that, I"
                " will input the same information of a single relation and all its attributes from the"
                " target schema."
            )
        },
        {
            "role": "user",
            "content": (
                "The attribute from the source schema is the following:\n"
                "Attribute name: '{{source_attribute.name}}'\n"
                "Attribute description: '{{source_attribute.description}}'\n"
                "Relation name: '{{source_relation.name}}'\n"
                "Relation description: '{{source_relation.description}}'"
            )
        },
        {
            "role": "user",
            "content": (
                "The relation from the target schema is the following:\n"
                "Relation name: '{{target_relation.name}}'\n"
                "Relation description: '{{target_relation.description}}'\n"
                "In the following, I will list all attributes of '{{target_relation.name}}'."
            )
        },
        {
            "role": "user",
            "content": (
                "Attribute name: '{{target_attribute.name}}'\n"
                "Attribute description: '{{target_attribute.description}}'"
            )
        },
        {
            "role": "user",
            "content": (
                "Explain which of the target attributes semantically match to '{{source_attribute.name}}'"
                " from '{{source_relation.name}}' of the source schema. Lets work this out step by step to"
                " make sure we get it correct. After your explanation, give a final decision"
                " JSON-formatted like this: `{ \"yes\": [], \"no\": [] }`. Under each of the following"
                " keys, list all target attributes of '{{target_relation.name}}' that apply: yes - if"
                " there is an invertible function that maps all values of the source attribute to the"
                " values of the target attribute; no - if there is no such function. Do not mention an"
                " attribute if there is not enough information to decide."
            )
        }
    ],
}

os.makedirs(os.getenv("TEMPLATE_DIR"), exist_ok=True)
for tpl_name in all_templates:
    with open(os.path.join(os.getenv("TEMPLATE_DIR"), f"{tpl_name}.json"), "w") as jsn:
        json.dump(templates[tpl_name.split("_")[-1]], jsn)

In [ ]:
prompts = {
    parameter.meta["path"]: build_prompts(parameters=parameter, templates=all_templates, modes=all_modes)
    for parameter in parameters
}
prompts = {
    param_path: [storage.store_prompt(prompt) for prompt in prompt_list]
    for param_path, prompt_list in prompts.items()
}
results = {}
start_idx = 0
total_parameters = len(parameters)

# Process prompts
for i, parameter in enumerate(parameters[start_idx:], start=1):
    print(f"Processing parameter {i}/{total_parameters}...")
    answers = await process_prompt_list(parameter, prompts[parameter.meta["path"]])
    results[parameter.meta["path"]] = storage.store_result(postprocess_answers(parameter, answers))
    print(f"Finished parameter {i}/{total_parameters}")


Processing parameter 1/37...
Processing 32 prompts in 1 chunks of size 64
Max concurrent batch operations: 4


Loading model Qwen/Qwen3-1.7B on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 403.75it/s]


Model loaded successfully.
Finished parameter 1/37


In [8]:
def result_to_dataframe(result: Result) -> pd.DataFrame:
    df_data = []
    for attribute_pair, result_pair in result.pairs.items():
        for vote in result_pair.votes:
            left_scope = "1" if len(vote.answer.attributes.sources) == 1 else "n"
            right_scope = "1" if len(vote.answer.attributes.targets) == 1 else "n"
            task_scope = left_scope + "-to-" + right_scope
            source = "cms." + result.parameters.source_relation.name + "." + attribute_pair.source.name #replace
            target = "omop." + result.parameters.target_relation.name + "." + attribute_pair.target.name #replace
            df_data.append({
                "task_scope": task_scope,
                "source": source.lower(),
                "source_attribute": attribute_pair.source.name,
                "source_relation": result.parameters.source_relation.name,
                "target": target.lower(),
                "target_attribute": attribute_pair.target.name,
                "target_relation": result.parameters.target_relation.name,
                "decision": str(vote.vote),
                "decision_index": vote.answer.index,
            })
    return pd.DataFrame(df_data)


result_df = pd.DataFrame()
for result in results.values():
    result_df = pd.concat((result_df, result_to_dataframe(result)))

result_df = result_df.merge(benchmark[["source", "target", "benchmark"]], on=["source", "target"], how="left").copy()
result_df["benchmark"] = result_df["benchmark"].fillna(False)
result_df.to_csv("results2/huggingface_cms_results_1.7btester2.csv", index=False) #replace
result_df

,task_scope,source,source_attribute,source_relation,target,target_attribute,target_relation,decision,decision_index,benchmark
0,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.payer_plan_period.family_source_value,Family_source_value,Payer_plan_period,no,0,False
1,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.payer_plan_period.payer_concept_id,Payer_concept_id,Payer_plan_period,no,0,False
2,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.payer_plan_period.payer_plan_period_end_date,Payer_plan_period_end_date,Payer_plan_period,no,0,False
3,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.payer_plan_period.payer_plan_period_id,Payer_plan_period_id,Payer_plan_period,no,0,False
4,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.payer_plan_period.payer_plan_period_start...,Payer_plan_period_start_date,Payer_plan_period,no,0,False
...,...,...,...,...,...,...,...,...,...,...
539,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.payer_plan_period.sponsor_source_concept_id,Sponsor_source_concept_id,Payer_plan_period,no,0,False
540,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.payer_plan_period.sponsor_source_value,Sponsor_source_value,Payer_plan_period,no,0,False
541,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.payer_plan_period.stop_reason_concept_id,Stop_reason_concept_id,Payer_plan_period,no,0,False
542,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.payer_plan_period.stop_reason_source_conc...,Stop_reason_source_concept_id,Payer_plan_period,no,0,False
